# Pre-Arrears Strategy — Component: DDR Strategy Target Import (SAS Step 3)

**Source**: `Pre-Arrears.txt`, lines 299–314 (`%macro ddr; ... %mend ddr; %ddr;`)

Third component of the 10-step strategy, converted independently. Unlike
Steps 1–2, this step does **not** query the `P_DW` warehouse — it reads a
pre-built external dataset (`ddr.ddr<yymm>`) produced by a separate DDR
("Direct Debit Reinstatement"? — see `TODO: VERIFY` below) strategy
process, and folds its `Target=1` accounts in as a single flag.


## 1. What the SAS component does

```sas
%let yymm   = intnx(month,&rundate,1)  formatted yymmn4.  -> the month AFTER &rundate, "YYMM"
%let yyyymm = intnx(month,&rundate,0)  formatted yymmn6.   -> the reporting month, "YYYYMM"

data DDR&yyyymm;
    set ddr.ddr&yymm(keep=PrimaryAccount Target);
    AccountNumber = PrimaryAccount;  format AccountNumber 9.;
    DDR = 1;                          format DDR 1.;
    if Target = 1;
    keep AccountNumber DDR;
run;

proc sort data=DDR&yyyymm nodup;
    by AccountNumber;
run;
```

- The source table name is built from **the month following** the
  reporting snapshot (`&rundate + 1 month`), per the code comment:
  *"Accounts flagged the following month of the observed snapshot"*.
  It is formatted `yymmn4.` — a bare **4-digit `YYMM`** (2-digit year +
  2-digit month, e.g. `2608` for August 2026) — which is a different
  format from the `yymmn6.` (`YYYYMM`) used for the output table name.
  Getting either of these formats or offsets wrong silently points the
  step at the wrong external table.
- Only `PrimaryAccount` and `Target` are read from the source (`keep=`
  on the `set` statement — the SAS equivalent of `SELECT
  PrimaryAccount, Target`, not `SELECT *`).
- `PrimaryAccount` is renamed to `AccountNumber`; a constant flag
  `DDR = 1` is added; rows are filtered to `Target = 1`; the output
  keeps only `AccountNumber` and `DDR`.
- `proc sort ... nodup` sorts by `AccountNumber` and removes **exact
  duplicate rows** (both `AccountNumber` and `DDR` identical to the
  row before it, post-sort) — not `nodupkey`, which would drop on the
  BY variable alone. Because `DDR` is constant `1` for every surviving
  row, the practical effect here is identical to a dedup on
  `AccountNumber` — but this distinction matters in general and is
  called out below.


## 2. Semantic risks

1. **Unknown external data source.** `ddr.ddr<yymm>` is read via a SAS
   `libname` (`ddr`) that is not defined anywhere in the supplied
   source excerpt — it could be a SAS dataset on a shared drive, a
   separate SQL Server database, a CSV drop, or another vendor feed.
   **TODO: VERIFY the physical location, connection method, and refresh
   cadence of `ddr.ddr<yymm>`** before wiring the loader below to a real
   source. The transformation logic (rename/flag/filter/dedup/sort) is
   fully converted and tested against representative sample data; only
   the *extraction* boundary is deliberately left pluggable pending that
   confirmation — this avoids inventing a connection this component was
   never shown.
2. **`DDR` acronym.** Not expanded anywhere in the source or comments.
   **TODO: VERIFY** what "DDR" stands for and what business process
   produces `ddr.ddr<yymm>`, so the flag's meaning is documented
   correctly for anyone using the converted output.
3. **Month-offset / format mismatch risk.** The source table month
   (`&rundate + 1`, `yymmn4.`) and the output table month (`&rundate + 0`,
   `yymmn6.`) use *different* offsets and *different* formats in the
   same macro. This is easy to transcribe incorrectly; both are
   implemented as separate, independently-tested functions below rather
   than one shared date helper, to keep the off-by-one-month and
   format differences explicit and visible in code.
4. **Missing vs. non-target.** `if Target = 1;` drops any row where
   `Target` is SAS-missing, `0`, or any value other than `1` — a
   missing `Target` is treated as "not a DDR target", not as an error
   or an inclusion. Pandas' `== 1` comparison against `NaN` already
   evaluates to `False` and behaves identically, but this is validated
   explicitly rather than assumed.
5. **`nodup` vs `nodupkey`.** `nodup` only drops a row if it is a
   complete duplicate of the immediately preceding row (all columns,
   not just the BY variable). If `AccountNumber` could ever appear with
   two *different* `DDR` values, `nodup` would keep both rows — unlike
   `nodupkey`, which would arbitrarily drop one. Since `DDR` is always
   `1` here, the observable result is the same as a distinct-by-account
   dedup, but the Python implementation replicates the actual `nodup`
   semantics (dedup on the full retained row set), not a
   `drop_duplicates(subset="AccountNumber")` shortcut, in case a future
   change to this step ever adds a second output column.
6. **Formats `9.` / `1.` are display-only.** They do not truncate or
   alter the underlying value of `AccountNumber` or `DDR` and are not
   reproduced as data transformations, only as documentation.
7. **Sort order.** Final output is sorted ascending by `AccountNumber`
   — preserved explicitly in the Python version rather than relying on
   join order downstream.
8. **Retained values / `first.`/`last.`.** Not applicable — no `retain`
   statement and no BY-group first/last logic in this component.


In [ ]:
"""
ddr_target_import.py
Production module: Pre-Arrears Strategy — Step 3: DDR Strategy Target Import.

Converted 1:1 from Pre-Arrears.txt lines 299-314 (SAS macro %ddr).

The extraction boundary (reading ddr.ddr<yymm>) is deliberately pluggable
via an injected `reader` callable, since the physical source of that
external dataset is not defined in the supplied excerpt (see risk #1).
"""

import logging
import sys
from dataclasses import dataclass
from datetime import date
from typing import Callable

import pandas as pd

logger = logging.getLogger("pre_arrears.ddr_target_import")
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-8s | %(name)s | %(message)s")
    )
    logger.addHandler(_handler)
    logger.setLevel(logging.INFO)


class DDRTargetImportError(Exception):
    """Raised when the Step 3 extraction/transform cannot be trusted to proceed."""


In [ ]:
@dataclass(frozen=True)
class DDRTargetImportConfig:
    rundate: date  # must be the 1st day of a calendar month

    def __post_init__(self):
        if self.rundate.day != 1:
            raise ValueError(f"rundate must be the first day of a month; got {self.rundate}")


In [ ]:
def get_reporting_period(rundate: date) -> int:
    """Equivalent of SAS: intnx(month,&rundate,0) formatted yymmn6. -> int YYYYMM."""
    return rundate.year * 100 + rundate.month


def get_ddr_source_month(rundate: date) -> str:
    """
    Equivalent of SAS: intnx(month,&rundate,1) formatted yymmn4.
    -> the month AFTER rundate, as a 4-character "YYMM" string
    (2-digit year, 2-digit month; no century, no separators).
    """
    total_months = rundate.year * 12 + (rundate.month - 1) + 1
    year, month = divmod(total_months, 12)
    month += 1  # divmod month index is 0-based
    return f"{year % 100:02d}{month:02d}"


def build_ddr_source_table_name(rundate: date) -> str:
    """Mirrors the SAS dataset reference `ddr.ddr&yymm` for documentation/logging."""
    return f"ddr.ddr{get_ddr_source_month(rundate)}"


def build_ddr_output_table_name(rundate: date) -> str:
    """Mirrors the SAS dataset name `DDR&yyyymm` for documentation/logging."""
    return f"DDR{get_reporting_period(rundate)}"


In [ ]:
REQUIRED_RAW_COLUMNS = ["PrimaryAccount", "Target"]


def transform_ddr_target_extract(raw: pd.DataFrame) -> pd.DataFrame:
    """
    Faithful conversion of the SAS DATA step + PROC SORT (lines 304-311).
    Input: raw extract with (at least) PrimaryAccount, Target columns
    (the SAS `keep=` list on the `set` statement selects only these two
    — the equivalent of `SELECT PrimaryAccount, Target`, never `SELECT *`).
    Output: one row per AccountNumber with DDR=1, sorted ascending.
    """
    missing_cols = [c for c in REQUIRED_RAW_COLUMNS if c not in raw.columns]
    if missing_cols:
        raise DDRTargetImportError(f"DDR source extract missing expected column(s): {missing_cols}")

    # SELECT PrimaryAccount, Target  (never SELECT *)
    working = raw[REQUIRED_RAW_COLUMNS].copy()

    # AccountNumber = PrimaryAccount;  DDR = 1;
    working["AccountNumber"] = working["PrimaryAccount"]
    working["DDR"] = 1

    # if Target = 1;  (missing/NaN Target correctly evaluates to False here, matching SAS)
    working = working.loc[working["Target"] == 1]

    # keep AccountNumber DDR;
    working = working[["AccountNumber", "DDR"]]

    # proc sort ... nodup; by AccountNumber;
    # `nodup` drops a row only if it exactly matches the immediately preceding
    # row post-sort across ALL retained columns (not just the BY variable) --
    # replicated here as a full-row drop_duplicates after sorting, rather
    # than a subset=["AccountNumber"] shortcut (see risk #5).
    working = working.sort_values("AccountNumber", kind="stable").reset_index(drop=True)
    working = working.drop_duplicates(keep="first").reset_index(drop=True)

    return working


In [ ]:
def validate_ddr_targets(df: pd.DataFrame) -> None:
    """Data-quality gate. Raises on any check failure; never silently mutates output."""
    required = ["AccountNumber", "DDR"]
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise DDRTargetImportError(f"Missing expected output column(s): {missing_cols}")

    if df.empty:
        logger.warning("DDR target import produced 0 rows. Confirm this is expected before proceeding.")
        return

    null_accounts = int(df["AccountNumber"].isna().sum())
    if null_accounts > 0:
        raise DDRTargetImportError(f"{null_accounts} null AccountNumber value(s) in DDR target output.")

    non_one_ddr = int((df["DDR"] != 1).sum())
    if non_one_ddr > 0:
        raise DDRTargetImportError(f"{non_one_ddr} row(s) with DDR != 1; DDR is expected to be a constant flag.")

    dup_accounts = int(df["AccountNumber"].duplicated().sum())
    if dup_accounts > 0:
        raise DDRTargetImportError(
            f"{dup_accounts} duplicate AccountNumber row(s) survived dedup — "
            f"check for an unexpected second distinct DDR value per account (see risk #5)."
        )

    is_sorted = df["AccountNumber"].is_monotonic_increasing
    if not is_sorted:
        raise DDRTargetImportError("Output is not sorted ascending by AccountNumber.")

    logger.info("Validation passed: %s DDR target account(s)", len(df))


In [ ]:
def load_ddr_target_extract(
    cfg: DDRTargetImportConfig,
    reader: Callable[[str], pd.DataFrame],
) -> pd.DataFrame:
    """
    Main entry point. Equivalent to SAS `%ddr;`.

    `reader` is an injected callable: given the source table name (e.g.
    "ddr.ddr2608"), it must return a DataFrame containing at least
    PrimaryAccount and Target columns. This keeps the (currently
    unconfirmed — see risk #1) I/O mechanism swappable without touching
    the business logic in `transform_ddr_target_extract`.
    """
    source_table = build_ddr_source_table_name(cfg.rundate)
    output_table = build_ddr_output_table_name(cfg.rundate)
    logger.info("Starting DDR Strategy Target Import: source=%s -> output=%s", source_table, output_table)

    try:
        raw = reader(source_table)
    except Exception as exc:  # noqa: BLE001
        raise DDRTargetImportError(f"Failed to read DDR source extract '{source_table}'") from exc

    result = transform_ddr_target_extract(raw)
    validate_ddr_targets(result)

    logger.info("Completed DDR Strategy Target Import: %s rows", len(result))
    return result


## 3. Example invocation

Not executed automatically (no confirmed real source to read from — see
risk #1). Shown here with a mock in-memory reader for illustration; a
production `reader` would be swapped in once the physical source of
`ddr.ddr<yymm>` is confirmed.


In [ ]:
RUN_LIVE_EXTRACTION = False  # flip to True once a real `reader` is available

if RUN_LIVE_EXTRACTION:
    def _mock_reader(table_name: str) -> pd.DataFrame:
        logger.info("Reading %s", table_name)
        return pd.DataFrame({"PrimaryAccount": [100001, 100002, 100003], "Target": [1, 0, 1]})

    config = DDRTargetImportConfig(rundate=date(2026, 7, 1))
    ddr_dataframe = load_ddr_target_extract(config, reader=_mock_reader)
    display(ddr_dataframe)
else:
    logger.info("RUN_LIVE_EXTRACTION is False — skipping extraction against a real source.")


## 4. Unit tests

In [ ]:
import unittest
from datetime import date as _date


class TestDateHelpers(unittest.TestCase):
    def test_reporting_period_basic(self):
        self.assertEqual(get_reporting_period(_date(2026, 7, 1)), 202607)

    def test_ddr_source_month_same_year(self):
        self.assertEqual(get_ddr_source_month(_date(2026, 7, 1)), "2608")  # Aug 2026

    def test_ddr_source_month_year_rollover(self):
        self.assertEqual(get_ddr_source_month(_date(2026, 12, 1)), "2701")  # Jan 2027

    def test_ddr_source_month_two_digit_month_padding(self):
        self.assertEqual(get_ddr_source_month(_date(2026, 1, 1)), "2602")  # Feb 2026, not "262"

    def test_source_table_name(self):
        self.assertEqual(build_ddr_source_table_name(_date(2026, 7, 1)), "ddr.ddr2608")

    def test_output_table_name(self):
        self.assertEqual(build_ddr_output_table_name(_date(2026, 7, 1)), "DDR202607")


class TestConfigValidation(unittest.TestCase):
    def test_rejects_non_first_of_month(self):
        with self.assertRaises(ValueError):
            DDRTargetImportConfig(rundate=_date(2026, 7, 15))


class TestTransform(unittest.TestCase):
    def test_filters_to_target_equals_one(self):
        raw = pd.DataFrame({"PrimaryAccount": [1, 2, 3], "Target": [1, 0, 1]})
        result = transform_ddr_target_extract(raw)
        self.assertEqual(sorted(result["AccountNumber"].tolist()), [1, 3])

    def test_missing_target_is_excluded_not_erroring(self):
        raw = pd.DataFrame({"PrimaryAccount": [1, 2], "Target": [1, None]})
        result = transform_ddr_target_extract(raw)
        self.assertEqual(result["AccountNumber"].tolist(), [1])

    def test_ddr_flag_is_always_one(self):
        raw = pd.DataFrame({"PrimaryAccount": [1, 2], "Target": [1, 1]})
        result = transform_ddr_target_extract(raw)
        self.assertTrue((result["DDR"] == 1).all())

    def test_output_columns_are_exactly_accountnumber_and_ddr(self):
        raw = pd.DataFrame({"PrimaryAccount": [1], "Target": [1]})
        result = transform_ddr_target_extract(raw)
        self.assertEqual(list(result.columns), ["AccountNumber", "DDR"])

    def test_output_sorted_ascending(self):
        raw = pd.DataFrame({"PrimaryAccount": [3, 1, 2], "Target": [1, 1, 1]})
        result = transform_ddr_target_extract(raw)
        self.assertEqual(result["AccountNumber"].tolist(), [1, 2, 3])

    def test_exact_duplicate_rows_deduped(self):
        raw = pd.DataFrame({"PrimaryAccount": [1, 1, 2], "Target": [1, 1, 1]})
        result = transform_ddr_target_extract(raw)
        self.assertEqual(result["AccountNumber"].tolist(), [1, 2])

    def test_missing_required_column_raises(self):
        raw = pd.DataFrame({"PrimaryAccount": [1]})  # no Target column
        with self.assertRaises(DDRTargetImportError):
            transform_ddr_target_extract(raw)

    def test_empty_extract_returns_empty_frame(self):
        raw = pd.DataFrame({"PrimaryAccount": [], "Target": []})
        result = transform_ddr_target_extract(raw)
        self.assertEqual(len(result), 0)
        self.assertEqual(list(result.columns), ["AccountNumber", "DDR"])

    def test_all_non_target_rows_returns_empty_frame(self):
        raw = pd.DataFrame({"PrimaryAccount": [1, 2], "Target": [0, 0]})
        result = transform_ddr_target_extract(raw)
        self.assertEqual(len(result), 0)


class TestValidation(unittest.TestCase):
    def test_valid_frame_passes(self):
        df = pd.DataFrame({"AccountNumber": [1, 2, 3], "DDR": [1, 1, 1]})
        validate_ddr_targets(df)  # should not raise

    def test_empty_frame_does_not_raise(self):
        df = pd.DataFrame({"AccountNumber": pd.Series(dtype="int64"), "DDR": pd.Series(dtype="int64")})
        validate_ddr_targets(df)  # should not raise, only warns

    def test_null_account_number_raises(self):
        df = pd.DataFrame({"AccountNumber": [1, None], "DDR": [1, 1]})
        with self.assertRaises(DDRTargetImportError):
            validate_ddr_targets(df)

    def test_non_one_ddr_value_raises(self):
        df = pd.DataFrame({"AccountNumber": [1, 2], "DDR": [1, 2]})
        with self.assertRaises(DDRTargetImportError):
            validate_ddr_targets(df)

    def test_duplicate_account_number_raises(self):
        df = pd.DataFrame({"AccountNumber": [1, 1], "DDR": [1, 1]})
        with self.assertRaises(DDRTargetImportError):
            validate_ddr_targets(df)

    def test_unsorted_output_raises(self):
        df = pd.DataFrame({"AccountNumber": [2, 1], "DDR": [1, 1]})
        with self.assertRaises(DDRTargetImportError):
            validate_ddr_targets(df)


class TestEndToEndLoad(unittest.TestCase):
    def test_load_with_injected_reader(self):
        def fake_reader(table_name):
            self.assertEqual(table_name, "ddr.ddr2608")
            return pd.DataFrame({"PrimaryAccount": [5, 4], "Target": [1, 1]})

        cfg = DDRTargetImportConfig(rundate=_date(2026, 7, 1))
        result = load_ddr_target_extract(cfg, reader=fake_reader)
        self.assertEqual(result["AccountNumber"].tolist(), [4, 5])

    def test_reader_failure_wrapped_in_ddr_error(self):
        def broken_reader(table_name):
            raise RuntimeError("connection refused")

        cfg = DDRTargetImportConfig(rundate=_date(2026, 7, 1))
        with self.assertRaises(DDRTargetImportError):
            load_ddr_target_extract(cfg, reader=broken_reader)


suite = unittest.TestLoader().loadTestsFromModule(sys.modules[__name__])
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Unit tests failed — see output above."


## 5. Source-to-target reconciliation

In [ ]:
RECONCILIATION_QUERIES = {
    # Applicable once both legacy and Python outputs are staged into a
    # queryable location (e.g. both loaded to staging tables for a UAT run).
    "control_totals": """
        SELECT 'legacy_sas' AS source, COUNT(*) AS row_count, COUNT(DISTINCT AccountNumber) AS distinct_accounts
        FROM staging.DDR_{yyyymm}_legacy
        UNION ALL
        SELECT 'python' AS source, COUNT(*) AS row_count, COUNT(DISTINCT AccountNumber) AS distinct_accounts
        FROM staging.DDR_{yyyymm}_python;
    """,
    "accounts_in_legacy_not_python": """
        SELECT l.AccountNumber
        FROM staging.DDR_{yyyymm}_legacy l
        LEFT JOIN staging.DDR_{yyyymm}_python p ON l.AccountNumber = p.AccountNumber
        WHERE p.AccountNumber IS NULL;
    """,
    "accounts_in_python_not_legacy": """
        SELECT p.AccountNumber
        FROM staging.DDR_{yyyymm}_python p
        LEFT JOIN staging.DDR_{yyyymm}_legacy l ON p.AccountNumber = l.AccountNumber
        WHERE l.AccountNumber IS NULL;
    """,
}


def render_reconciliation_queries(yyyymm: int) -> dict:
    return {name: sql.format(yyyymm=yyyymm) for name, sql in RECONCILIATION_QUERIES.items()}


def reconcile_dataframes(legacy_df: pd.DataFrame, python_df: pd.DataFrame) -> dict:
    """In-memory reconciliation for a test/UAT run where both outputs are already DataFrames."""
    merged = legacy_df.merge(python_df, on="AccountNumber", how="outer", indicator=True)
    return {
        "control_totals": pd.DataFrame([
            {"source": "legacy", "row_count": len(legacy_df), "distinct_accounts": legacy_df["AccountNumber"].nunique()},
            {"source": "python", "row_count": len(python_df), "distinct_accounts": python_df["AccountNumber"].nunique()},
        ]),
        "accounts_in_legacy_not_python": merged.loc[merged["_merge"] == "left_only", ["AccountNumber"]],
        "accounts_in_python_not_legacy": merged.loc[merged["_merge"] == "right_only", ["AccountNumber"]],
    }


## 6. Consolidated `TODO: VERIFY` list

- **Physical source of `ddr.ddr<yymm>`** — connection method, refresh
  timing relative to `&rundate`, and whether it is guaranteed to exist
  for every reporting month (what should happen if the following
  month's DDR extract hasn't landed yet when this strategy runs?).
- **Meaning of "DDR"** — confirm the business definition so the `DDR`
  flag is documented accurately for downstream consumers.
- **Column types in the source extract** — confirm whether
  `PrimaryAccount`/`AccountNumber` is numeric or character in the real
  source (SAS numeric-to-numeric assignment here implies numeric, but
  this could not be confirmed without the `ddr` library's format
  catalog).
